# Visual Contrastive Decoding (VCD) - CHAIR Benchmark on Kaggle

This notebook runs the **CHAIR (Caption Hallucination Assessment with Image Relevance)** benchmark on **500 images sampled from COCO val2014 (seed: 2027)** using **Visual Contrastive Decoding (VCD)** on Kaggle with free **GPU 2xT4**.

### Benchmark Settings & Constraints:
- **Evaluation Dataset**: 500 images sampled from COCO val2014 with **Seed: 2027** (`selected_chair_val2014_seed2027.json`)
- **Prompt**: `"Describe this image."`
- **Generation**: `max_new_tokens = 128`, Greedy Decoding (`do_sample = False`, `temperature = 0.0`)
- **Evaluation Code**: Standalone CHAIR metric ([Maxlinn/CHAIR-metric-standalone](https://github.com/Maxlinn/CHAIR-metric-standalone/tree/main))
- **Reported Metrics**:
  1. `CHAIRs` (%): Sentence-level hallucination rate
  2. `CHAIRi` (%): Instance-level hallucination rate
  3. `Recall` (%): Ground-truth object recall
  4. `Caption Length`: Average caption length (words)
- **Supported Models**:
  - `llava` (`llava-hf/llava-1.5-7b-hf`)
  - `qwen2vl` (`Qwen/Qwen2-VL-7B-Instruct`)
- **Methods**: Visual Contrastive Decoding (`--use_vcd`) vs Standard Baseline (`--no_vcd`)

### Prerequisites Before Running:
1. **Accelerator**: Select **GPU T4 x2** (Notebook Settings -> Accelerator).
2. **Internet**: Toggle **Internet ON** (Notebook Settings -> Internet).
3. **HuggingFace Token**: Add a secret named `HF_TOKEN` in **Add-ons -> Secrets**.
4. **COCO 2014 val images**: Add dataset in **+ Add Input** (e.g. `datasets/biminhco/val2014/val2014` or search `val2014`).


In [ ]:
# Cell 1: Environment Setup & Dependencies Installation
# 1. Gỡ bỏ torchaudio để giải quyết triệt để xung đột CUDA version mismatch
!pip uninstall -y -q torchaudio

# 2. Cài đặt các thư viện cần thiết
!pip install -q --no-cache-dir \
    "transformers>=4.45.0" \
    "accelerate>=0.26.0" \
    sentencepiece \
    protobuf \
    tiktoken \
    qwen_vl_utils \
    pyyaml \
    tqdm \
    nltk \
    huggingface_hub \
    pandas

print("✅ Dependencies successfully installed!")
from transformers import AutoProcessor
print("✅ AutoProcessor import verified successfully!")


In [ ]:
# Cell 2: Authenticate with Hugging Face via Kaggle Secrets
import os
from kaggle_secrets import UserSecretsClient
import huggingface_hub

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("✅ Successfully logged in to Hugging Face Hub!")
except Exception as e:
    print(f"[Notice] Kaggle Secrets login info: {e}")
    print("If accessing gated models, ensure 'HF_TOKEN' is added in Add-ons -> Secrets.")


In [ ]:
# Cell 3: Clone Repository or Pull Latest Code
import os

repo_path = "/kaggle/working/VCD"

if not os.path.exists(repo_path):
    !git clone https://github.com/ntmy12/VCD.git {repo_path}
else:
    print(f"Pulling latest changes in {repo_path}...")
    !cd {repo_path} && git pull origin master

# Enter vcd_experiments directory
%cd /kaggle/working/VCD/vcd_experiments


In [ ]:
# Cell 4: Hardware & Environment Verification
import torch
import yaml
import os

print("=== GPU Environment ===")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        print(f"  Device {i}: {prop.name} ({prop.total_memory / 1e9:.2f} GB VRAM)")

print("\n=== CHAIR Verification ===")
config_file = "configs/data_paths_kaggle.yaml"
with open(config_file, "r") as f:
    cfg = yaml.safe_load(f)

coco_img_dir = cfg.get("coco_val2014_images", "")
manifest_path = cfg.get("chair_manifest", "benchmarks/chair/selected_chair_val2014_seed2027.json")
cache_path = cfg.get("chair_eval_cache", "benchmarks/chair/chair.pkl")

print(f"Configured Image Dir: {coco_img_dir}")
print(f"Manifest Path:        {manifest_path} (Exists: {os.path.isfile(manifest_path)})")
print(f"Evaluator Cache:      {cache_path} (Exists: {os.path.isfile(cache_path)})")


In [ ]:
# Cell 5: Run CHAIR Benchmark Evaluation
# Select model: 'llava' or 'qwen2vl'
MODEL = "llava"

# Evaluation settings:
# --use_vcd: enable Visual Contrastive Decoding (use --no_vcd for baseline)
# --seed 2027: loads the pre-computed 500 images from COCO val2014
# --max_new_tokens 128: generates up to 128 tokens per caption
# --prompt "Describe this image.": standard prompt for CHAIR
!python benchmarks/chair/run_chair.py \
    --model {MODEL} \
    --use_vcd \
    --config_path configs/data_paths_kaggle.yaml \
    --seed 2027 \
    --num_samples 500 \
    --max_new_tokens 128 \
    --prompt "Describe this image." \
    --noise_step 500 \
    --cd_alpha 1.0 \
    --cd_beta 0.1


In [ ]:
# Cell 6: Display CHAIR Results & Sample Outputs
import glob
import json
import pandas as pd

run_dirs = sorted(glob.glob("results/*chair*"))
if run_dirs:
    latest_dir = run_dirs[-1]
    metrics_file = os.path.join(latest_dir, "metrics.json")
    config_file = os.path.join(latest_dir, "run_config.json")
    
    if os.path.exists(config_file):
        with open(config_file, "r") as f:
            cfg = json.load(f)
        print(f"Run Directory: {latest_dir}")
        print(f"Model: {cfg.get('model')} | VCD: {cfg.get('use_vcd')} | Max Tokens: {cfg.get('max_new_tokens')} | Seed: {cfg.get('seed')}")

    if os.path.exists(metrics_file):
        with open(metrics_file, "r") as f:
            metrics = json.load(f)
        df = pd.DataFrame([metrics])
        df.columns = ["CHAIRs (%)", "CHAIRi (%)", "Recall (%)", "Caption Length (words)", "Total Images"]
        print(f"\n{'='*30} CHAIR BENCHMARK METRICS {'='*30}")
        display(df)
    
    raw_file = os.path.join(latest_dir, "raw_outputs.jsonl")
    if os.path.exists(raw_file):
        print(f"\n{'='*30} SAMPLE GENERATED CAPTIONS {'='*30}")
        with open(raw_file, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5: break
                item = json.loads(line)
                print(f"[{i+1}] Image {item['file_name']} (ID: {item['image_id']}):")
                print(f"    Caption: {item['caption']}\n")
else:
    print("No results found in results/ directory yet.")
